# Results — FID vs. Sampling Steps

Reproduces every figure in the report end to end: trains Flow Matching and DDPM (and,
optionally, a reflowed model), then measures **FID as a function of the number of
sampling steps**. Runs top-to-bottom on a Kaggle T4 (~4–5 h). Set `DATASET` below to
`mnist` or `fashion` — running both reproduces the domain-shift comparison.

## Setup

In [ ]:
import torch
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())

In [ ]:
# On Kaggle this clones the repo; locally (from the repo root) it is a no-op.
import os
if not os.path.isdir('flow-matching-fewstep') and not os.path.isdir('fmfs'):
    !git clone -q https://github.com/xHansi/flow-matching-fewstep.git
REPO = os.path.abspath('flow-matching-fewstep') if os.path.isdir('flow-matching-fewstep') else os.getcwd()
!pip install -q pytorch-fid
print('repo at', REPO)

In [ ]:
DATASET = 'mnist'   # or 'fashion' for the domain-shift run
print('dataset:', DATASET)

## 1. Train both methods

Each run saves a checkpoint, a loss curve and a class-conditional preview grid under
`results/<method>_<dataset>/`. Bump `--epochs` for the final numbers.

In [ ]:
!cd {REPO} && PYTHONPATH={REPO} python -m scripts.train --method flow --dataset {DATASET} --epochs 30

In [ ]:
!cd {REPO} && PYTHONPATH={REPO} python -m scripts.train --method ddpm --dataset {DATASET} --epochs 30

## 2. Reflow the Flow Matching model

Retrains the flow model on its own `(noise, sample)` pairs to straighten the ODE paths,
which helps at 1–2 steps.

In [ ]:
!cd {REPO} && PYTHONPATH={REPO} python -m scripts.reflow --ckpt results/flow_{DATASET}/ckpt.pt --pairs 50000 --gen-steps 100 --epochs 30

## 3. Loss curves and preview grids

In [ ]:
from IPython.display import Image, display
for m in ('flow', 'ddpm', 'reflow'):
    d = f'{REPO}/results/{m}_{DATASET}'
    print(m); display(Image(f'{d}/loss_curve.png')); display(Image(f'{d}/samples.png'))

## 4. FID vs. steps (the core result)

Sweeps the step count for each checkpoint and writes `results/fid/` (a JSON plus one plot
per metric). Measured **without** guidance (`--cfg-scale 1.0`): guidance inflates FID and
over-saturates DDPM at high step counts, which would confound the steps comparison.
Two metrics — `inception` (standard InceptionV3-FID) and `classifier` (a small CNN
trained on this dataset, i.e. domain-matched).

In [ ]:
!cd {REPO} && PYTHONPATH={REPO} python -m scripts.eval_fid \
    --ckpts flow=results/flow_{DATASET}/ckpt.pt ddpm=results/ddpm_{DATASET}/ckpt.pt reflow=results/reflow_{DATASET}/ckpt.pt \
    --dataset {DATASET} --steps 1,2,4,8,16,50,100 --n-samples 5000 --cfg-scale 1.0 --metric both

In [ ]:
import json
fid = json.load(open(f'{REPO}/results/fid/fid.json'))
steps = [1, 2, 4, 8, 16, 50, 100]
for metric, per_method in fid.items():
    print(f'\n=== {metric}-FID (lower is better) ===')
    print('steps :', '  '.join(f'{s:>6}' for s in steps))
    for name, d in per_method.items():
        print(f'{name:6s}:', '  '.join(f'{d[str(s)]:6.1f}' for s in steps))

In [ ]:
for metric in ('inception', 'classifier'):
    display(Image(f'{REPO}/results/fid/fid_vs_steps_{metric}.png'))

## 5. Sample grids: few vs. many steps

A visual read of the same story — Flow stays coherent at 2 steps where DDPM does not.

In [ ]:
for m in ('flow', 'ddpm', 'reflow'):
    ck = f'results/{m}_{DATASET}'
    !cd {REPO} && PYTHONPATH={REPO} python -m scripts.sample --ckpt {ck}/ckpt.pt --steps 2,8,100 --per-class 10 --cfg-scale 2.0
    for s in (2, 8, 100):
        print(f'{m}: {s} steps'); display(Image(f'{REPO}/{ck}/samples_steps{s}.png'))

## 6. Domain-shift comparison

Once both datasets have been evaluated (run this notebook with `DATASET='mnist'` and
again with `'fashion'`, saving each `fid.json` under `figures/<dataset>/`), this builds
the side-by-side plot — the same figure used in the report.

In [ ]:
import os
have = [p for p in ('figures/mnist/fid.json', 'figures/fashion/fid.json') if os.path.exists(f'{REPO}/{p}')]
if len(have) == 2:
    !cd {REPO} && PYTHONPATH={REPO} python -m scripts.compare_datasets
    display(Image(f'{REPO}/figures/comparison/fid_vs_steps_inception.png'))
else:
    print('Run both datasets first; found:', have)